In [1]:
%pip install duckdb-extensions duckdb-extension-httpfs
from duckdb_extensions import import_extension;import duckdb;import_extension("httpfs")
con = duckdb.connect();con.execute("LOAD httpfs;")
import os,requests,time,pandas as pd,yaml;u="http://sql-exercise.sql-exercise-prod.svc.cluster.local/";f="py_workshop.py"
if os.path.exists(f):os.remove(f)
if not os.path.exists(f):
    r=requests.get(u+"get_py_workshop");r.status_code==200 and open(f,"wb").write(r.content) and time.sleep(1)
from py_workshop import *;runner.set_url(u);runner.logon();schema_name = runner.user_schema;schema_name = runner.user_schema;config = runner.get_config()
pd.set_option("display.max_columns", None);pd.set_option("display.width", 2000);pd.set_option("display.max_colwidth", None);pd.set_option("display.max_rows", None)

runner.course = 'de'
runner.material( 'de_spark_sql_hw', course_launch='de_2026')

Note: you may need to restart the kernel to use updated packages.
✅ Вы вошли как a.d.kulikov@centraluniversity.ru. Ваша схема в БД - a_d_kulikov. Ключ user id - 81cd7d474a43b47a49ea0ff060ce71e8


In [ ]:
import os
import json
import yaml
import socket
from datetime import date, timedelta, datetime

import duckdb
from duckdb_extensions import import_extension
import_extension("httpfs")
import pandas as pd
from pyspark.sql import SparkSession 
from pyspark.sql.functions import col, from_json, expr, to_timestamp, window, when, sum, count, avg, lit, max, min, to_utc_timestamp, to_date, current_timestamp, unix_timestamp, round, hour
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, FloatType, DoubleType
from pyspark.sql.window import Window

os.environ["PYSPARK_PYTHON"] = "/opt/spark/python3.11/versions/3.11.13/bin/python3"
os.environ["AWS_REGION"]="ru-central1"

con = duckdb.connect()
con.execute(f"""
    LOAD httpfs;
    SET s3_region='{os.environ['AWS_REGION']}';
    SET s3_endpoint='storage.yandexcloud.net';
    SET s3_access_key_id='{config['s3']['my']['access_key']}'; 
    SET s3_secret_access_key='{config['s3']['my']['secret_key']}';
    SET s3_use_ssl=true;
    SET s3_url_style='path';
""")
con.execute(f"SET s3_region='{os.environ['AWS_REGION']}';")

def new_spark()->SparkSession:
    spark = (
        SparkSession.builder

        # Название сессии
        .appName(f"Iceberg practice, {schema_name}") 
        .config("spark.ui.showConsoleProgress", "true") 
        .config("spark.jars.packages", "org.apache.spark:spark-connect_2.12:3.5.0,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.9.1,org.postgresql:postgresql:42.7.7,org.apache.iceberg:iceberg-aws-bundle:1.9.1,org.apache.spark:spark-sql-kafka-0-10_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.1,com.amazonaws:aws-java-sdk-bundle:1.11.704") 
        .config('spark.driver.host', socket.gethostbyname(socket.gethostname()))

        # Настройки подключения
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
        .config("spark.hadoop.fs.s3.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") 
        .config("spark.hadoop.fs.AbstractFileSystem.s3a.impl", "org.apache.hadoop.fs.s3a.S3A") 
        .config("spark.hadoop.fs.s3a.endpoint", "storage.yandexcloud.net") 
        .config("spark.hadoop.fs.s3a.endpoint.region", os.environ['AWS_REGION']) 
        .config("spark.hadoop.fs.s3a.path.style.access", "true") 
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") 
        .config("spark.hadoop.fs.s3a.bucket.culab-files-prod.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") 
        .config("spark.hadoop.fs.s3a.bucket.culab-files-prod.access.key", config['s3']['com']['access_key'])
        .config("spark.hadoop.fs.s3a.bucket.culab-files-prod.secret.key", config['s3']['com']['secret_key'])
        .config("spark.hadoop.fs.s3a.bucket.culab-student-files.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") 
        .config("spark.hadoop.fs.s3a.bucket.culab-student-files.access.key", config['s3']['my']['access_key'])
        .config("spark.hadoop.fs.s3a.bucket.culab-student-files.secret.key", config['s3']['my']['secret_key'])
        .config("spark.driverEnv.AWS_REGION", os.environ['AWS_REGION'])
        .config("spark.executorEnv.AWS_REGION", os.environ['AWS_REGION'])
        
        .getOrCreate()
    )
    return spark

spark = new_spark()

spark

In [ ]:
spark.stop()

### Импорт данных из источников

Источник: два дня записей о перелётах - ds_avia_sim_ru

Его структура:

 |-- flight_number: string (nullable = true)  
 |-- airline: string (nullable = true)  
 |-- departure_airport: string (nullable = true)  
 |-- arrival_airport: string (nullable = true)  
 |-- departure_time: timestamp (nullable = true)  
 |-- arrival_time: timestamp (nullable = true)  
 |-- status: string (nullable = true)  

### Задача 1. Создание представления в Spark
Необходимо выбрать данные о рейсах за два дня: s3a://culab-files-prod/files/public/data_engineering/data_streams/ds_avia_sim_ru. На основе этих данных требуется создать временное представление **avia** для последующего использования в Spark SQL.

Нужно написать любой запрос к представлению.

In [4]:
%%py_code_local cell_1_1
# Задача 1. Ячейка 1. Создание представления

# Устанавливаем даты для анализа
today = date.today()
d1 = (today - timedelta(days=1)).strftime("%Y-%m-%d")
d2 = (today - timedelta(days=2)).strftime("%Y-%m-%d")

# Пути к CSV за два дня
path = f"s3a://culab-files-prod/files/public/data_engineering/data_streams/ds_avia_sim_ru"
avia_paths = [
    f"{path}/{d1}/*.csv",
    f"{path}/{d2}/*.csv",
]

# Данные о рейсах
avia_df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(avia_paths)
)

# Указываем путь для сохранения результатов
target_path = f"s3a://culab-student-files/users/{schema_name}/spark_hw/"

# Создаём временное представление для хранения данных
avia_df.createOrReplaceTempView("avia")

▶️ Python | task=1
⏱️ start: 2026-04-20 18:16:51
Переменные сохранены
⏱️ end:   2026-04-20 18:17:15
✅ Done | task=1 | duration=24.179s


In [5]:
%%py_code_local cell_1_2
# Задача 1. Ячейка 2. Запрос к представлению

avia_df.printSchema()
avia_df.show(5, truncate=False)

▶️ Python | task=1
⏱️ start: 2026-04-20 18:17:34
root
 |-- flight_number: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- departure_airport: string (nullable = true)
 |-- arrival_airport: string (nullable = true)
 |-- departure_time: timestamp (nullable = true)
 |-- arrival_time: timestamp (nullable = true)
 |-- status: string (nullable = true)

+-------------+----------------+-----------------+---------------+-------------------+-------------------+---------+
|flight_number|airline         |departure_airport|arrival_airport|departure_time     |arrival_time       |status   |
+-------------+----------------+-----------------+---------------+-------------------+-------------------+---------+
|PO5022       |Pobeda          |IKT              |ROV            |2026-04-18 23:41:00|2026-04-19 04:16:00|delayed  |
|AE572        |Aeroflot        |UFA              |KZN            |2026-04-19 02:13:00|2026-04-19 04:25:00|delayed  |
|S79388       |S7 Airlines     |KHV           

### Задача 2. Количество рейсов, Spark SQL
Требуется при помощи Spark SQL определить количество рейсов для каждой авиакомпании и среднюю продолжительность полёта в часах.

На выходе требуются следующие столбцы:
- airline - Название авиакомпании
- flight_count - Количество рейсов
- avg_duration_hours - Средняя продолжительность полёта в часах (округлённая до 2 знаков)

Результаты надо отсортировать по flight_count по убыванию, затем по airline.

**Важная подсказка**: для решения потребуется конвертировать даты в UNIX_TIMESTAMP

Полученные данные необходимо сохранить в users/USER_SCHEMA/spark_hw/**flight_count.parquet**.

Напиши запрос к новому файлу (любой), чтобы проверить, что он записался корректно.



In [6]:
%%py_code_local cell_2_1
# Задача 2. Ячейка 1. Spark SQL. Запрос к исходным данным

result_df = spark.sql(f"""
    select
        airline,
        count(flight_number) as flight_count,
        round(avg(
            (unix_timestamp(arrival_time) - unix_timestamp(departure_time)) / 3600
          ), 2) as avg_duration_hours
    from avia
    group by airline
    order by flight_count desc, airline asc
""")

result_df.show(5, truncate=False)

▶️ Python | task=2
⏱️ start: 2026-04-20 18:18:25
+----------------+------------+------------------+
|airline         |flight_count|avg_duration_hours|
+----------------+------------+------------------+
|Pobeda          |825         |3.48              |
|Ural Airlines   |814         |3.38              |
|S7 Airlines     |798         |3.49              |
|Aeroflot        |779         |3.57              |
|Rossiya Airlines|776         |3.52              |
+----------------+------------+------------------+

Переменные сохранены
⏱️ end:   2026-04-20 18:18:36
✅ Done | task=2 | duration=11.808s


In [7]:
%%py_code_local cell_2_2
# Задача 2. Ячейка 2. Spark SQL. Сохранение в S3

(
    result_df
        .write
        .mode("overwrite")
        .parquet(target_path + 'flight_count.parquet')
)

▶️ Python | task=2
⏱️ start: 2026-04-20 18:18:48
Переменные сохранены
⏱️ end:   2026-04-20 18:19:04
✅ Done | task=2 | duration=15.565s


In [8]:
%%py_code_local cell_2_3
# Задача 2. Ячейка 3. Spark SQL. Запрос к новому файлу

(
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .parquet(target_path + 'flight_count.parquet')
).show(5, truncate=False)

▶️ Python | task=2
⏱️ start: 2026-04-20 18:19:31
+----------------+------------+------------------+
|airline         |flight_count|avg_duration_hours|
+----------------+------------+------------------+
|Pobeda          |825         |3.48              |
|Ural Airlines   |814         |3.38              |
|S7 Airlines     |798         |3.49              |
|Aeroflot        |779         |3.57              |
|Rossiya Airlines|776         |3.52              |
+----------------+------------+------------------+

Переменные сохранены
⏱️ end:   2026-04-20 18:19:33
✅ Done | task=2 | duration=1.567s


### Задача 3. Топ-3 популярных маршрута, Spark DataFrame API
Требуется при помощи Spark DataFrame API найти топ-3 самых популярных маршрута (пара аэропорт вылета — аэропорт прилёта) и количество рейсов на каждом маршруте.

На выходе требуются следующие столбцы:
- departure_airport - Аэропорт вылета
- arrival_airport - Аэропорт прилёта
- route_flights - Количество рейсов на маршруте

Результаты надо отсортировать по route_flights по убыванию и ограничить 3 строками.



In [9]:
%%py_code_local cell_3_1
# Задача 3. Ячейка 1. Spark SQL. Запрос

top3_routes_df = (
    avia_df
        .groupBy("departure_airport", "arrival_airport")
        .agg(count("*").alias("route_flights"))
        .orderBy(col("route_flights").desc())
        .limit(3)
)

top3_routes_df.show(truncate=False)

▶️ Python | task=3
⏱️ start: 2026-04-20 18:19:49
+-----------------+---------------+-------------+
|departure_airport|arrival_airport|route_flights|
+-----------------+---------------+-------------+
|KJA              |KRR            |54           |
|KJA              |LED            |48           |
|LED              |KJA            |47           |
+-----------------+---------------+-------------+

Переменные сохранены
⏱️ end:   2026-04-20 18:20:00
✅ Done | task=3 | duration=11.226s


### Задача 4. Статусы рейсов и агрегация
Требуется при помощи Spark DataFrame API или Spark SQL для каждого аэропорта вылета определить количество рейсов по каждому статусу и общее количество рейсов.

На выходе требуются следующие столбцы:
- departure_airport - Аэропорт вылета
- status - Статус рейса
- status_count - Количество рейсов с данным статусом
- total_flights - Общее количество рейсов из данного аэропорта

Результаты надо отсортировать по departure_airport, затем по status_count по убыванию.

**Важная подсказка**: для решения этой задачи могут пригодиться оконные функции.
Как их использовать, можно узнать здесь:
- Spark SQL: https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-window.html
- Spark DataFrame API: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.Window.html


Полученные данные необходимо сохранить в users/{schema_name}/spark_hw/**flight_status.parquet**.


Затем напиши запрос к сохраненному файлу, который вычислит процент рейсов в каждом статусе (status_prc, два знака после запятой) относительно общего количества рейсов из каждого аэропорта. Нужно вывести аэропорт, статус и новое поле.



In [10]:
%%py_code_local cell_4_1
# Задача 4. Ячейка 1. Spark SQL. Запрос к исходным данным

from pyspark.sql.window import Window

status_counts_df = (
    avia_df
        .groupBy("departure_airport", "status")
        .agg(count("*").alias("status_count"))
        .withColumn(
            "total_flights", 
            sum("status_count").over(
                Window.partitionBy("departure_airport")
            )
        )
        .orderBy("departure_airport", col("status_count").desc())
)

status_counts_df.show(5, truncate=False)

▶️ Python | task=4
⏱️ start: 2026-04-20 18:20:25
+-----------------+---------+------------+-------------+
|departure_airport|status   |status_count|total_flights|
+-----------------+---------+------------+-------------+
|DME              |cancelled|111         |276          |
|DME              |delayed  |83          |276          |
|DME              |departed |82          |276          |
|IKT              |departed |99          |277          |
|IKT              |cancelled|91          |277          |
+-----------------+---------+------------+-------------+
only showing top 5 rows

Переменные сохранены
⏱️ end:   2026-04-20 18:20:35
✅ Done | task=4 | duration=10.291s


In [11]:
%%py_code_local cell_4_2
# Задача 4. Ячейка 2. Spark SQL. Сохранение в S3

(
    status_counts_df
        .write
        .mode("overwrite")
        .parquet(target_path + 'flight_status.parquet')
)

▶️ Python | task=4
⏱️ start: 2026-04-20 18:21:37
Переменные сохранены
⏱️ end:   2026-04-20 18:21:49
✅ Done | task=4 | duration=12.276s


In [15]:
%%py_code_local cell_4_3
# Задача 4. Ячейка 3. Spark SQL. Запрос к новому файлу

saved_df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .parquet(target_path + 'flight_status.parquet')
)

percentage_df = (
    saved_df
    .withColumn(
        "status_prc", 
        round((col("status_count") / col("total_flights")) * 100, 2)
    )
    .select("departure_airport", "status", "status_prc")
)

percentage_df.show(5, truncate=False)

▶️ Python | task=4
⏱️ start: 2026-04-20 18:23:35
+-----------------+---------+----------+
|departure_airport|status   |status_prc|
+-----------------+---------+----------+
|DME              |cancelled|40.22     |
|DME              |delayed  |30.07     |
|DME              |departed |29.71     |
|IKT              |departed |35.74     |
|IKT              |cancelled|32.85     |
+-----------------+---------+----------+
only showing top 5 rows

Переменные сохранены
⏱️ end:   2026-04-20 18:23:36
✅ Done | task=4 | duration=0.587s


### Задача 5. Ночные рейсы
Требуется при помощи Spark SQL и отдельно Spark DataFrame API найти рейсы, которые вылетают поздно вечером (после 22:00 включительно) или рано утром (до 06:00 не включительно), и для каждого аэропорта вылета посчитать их процент от общего количества рейсов.

На выходе требуются следующие столбцы:
- departure_airport - Аэропорт вылета
- night_flights - Количество ночных рейсов (22:00-06:00)
- total_flights - Общее количество рейсов
- night_flight_ratio - Процент ночных рейсов (округлённый до 3 знаков)

Результаты надо отсортировать по night_flight_ratio по убыванию.



In [16]:
%%py_code_local cell_5_1
# Задача 5. Ячейка 1. Spark SQL. Запрос к исходным данным

(
    spark.sql("""
        with flights_cnt_cte as (
            select
                departure_airport
                , count(case when hour(departure_time) >= 22 or hour(departure_time) < 6 then 1 end) as night_flights
                , count(*) as total_flights
            from avia
            group by departure_airport
        )
        select
            *
            , round(night_flights / total_flights * 100, 3) as night_flight_ratio
        from flights_cnt_cte
        order by night_flight_ratio desc
    """)
).show(5, truncate=False)

▶️ Python | task=5
⏱️ start: 2026-04-20 18:29:02
+-----------------+-------------+-------------+------------------+
|departure_airport|night_flights|total_flights|night_flight_ratio|
+-----------------+-------------+-------------+------------------+
|VKO              |119          |294          |40.476            |
|KZN              |99           |281          |35.231            |
|DME              |97           |276          |35.145            |
|SVO              |83           |244          |34.016            |
|KJA              |166          |492          |33.74             |
+-----------------+-------------+-------------+------------------+
only showing top 5 rows

Переменные сохранены
⏱️ end:   2026-04-20 18:29:18
✅ Done | task=5 | duration=16.208s


In [17]:
%%py_code_local cell_5_2
# Задача 5. Ячейка 2. Spark DataFrame API. Запрос к исходным данным

from pyspark.sql.functions import hour

night_flights_df = (
    avia_df
        .withColumn(
            "is_night", 
            when(
                (hour("departure_time") >= 22) | (hour("departure_time") < 6),
                1
            ).otherwise(0)
        )
)

(
    night_flights_df
    .groupBy("departure_airport")
    .agg(
        sum("is_night").alias("night_flights"),
        count("*").alias("total_flights")
    )
    .withColumn(
        "night_flight_ratio",
        pyspark_round(
            col("night_flights") / col("total_flights") * 100,
            3
        )
    )
    .orderBy(col("night_flight_ratio").desc())
).show(5, truncate=False)

▶️ Python | task=5
⏱️ start: 2026-04-20 18:29:46
+-----------------+-------------+-------------+------------------+
|departure_airport|night_flights|total_flights|night_flight_ratio|
+-----------------+-------------+-------------+------------------+
|VKO              |119          |294          |40.476            |
|KZN              |99           |281          |35.231            |
|DME              |97           |276          |35.145            |
|SVO              |83           |244          |34.016            |
|KJA              |166          |492          |33.74             |
+-----------------+-------------+-------------+------------------+
only showing top 5 rows

Переменные сохранены
⏱️ end:   2026-04-20 18:29:58
✅ Done | task=5 | duration=11.654s
